# Chapter 5 Exercises: Loss Evaluation (Cross-Entropy & Perplexity)

Bu notebook, LLM'lerde **cross-entropy loss** ve **perplexity** metriklerini anlamak ve uygulamak için tasarlanmış egzersizler içerir.

In [1]:
import torch
import tiktoken
import math
from previous_chapters import GPTModel, create_dataloader_v1
import os
import requests

## Egzersiz 1: Manuel Cross-Entropy Hesaplama

In [2]:
inputs = torch.tensor([[16833, 3626, 6100], [40, 1107, 588]])
targets = torch.tensor([[3626, 6100, 345], [1107, 588, 11311]])

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, "context_length": 256, "emb_dim": 768,
    "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

with torch.no_grad():
    logits = model(inputs)
print(f"Logits shape: {logits.shape}")

Logits shape: torch.Size([2, 3, 50257])


### Softmax Probability Hesaplama

In [4]:
probas = torch.softmax(logits, dim=-1)
print(f"Probas shape: {probas.shape}")
print(f"Sum of probas for first token: {probas[0, 0].sum().item():.6f}")

Probas shape: torch.Size([2, 3, 50257])
Sum of probas for first token: 1.000000


### Target Probability Hesaplama

In [5]:
target_probas_1 = probas[0, [0, 1, 2], targets[0]]
target_probas_2 = probas[1, [0, 1, 2], targets[1]]
print("Text 1 target probabilities:", target_probas_1)
print("Text 2 target probabilities:", target_probas_2)

Text 1 target probabilities: tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])
Text 2 target probabilities: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


### Log Probabilities ve Cross-Entropy

In [6]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
neg_log_probas = -log_probas
avg_neg_log_probas = torch.mean(neg_log_probas)
print(f"Manuel Cross-Entropy Loss: {avg_neg_log_probas:.4f}")

Manuel Cross-Entropy Loss: 10.7940


### PyTorch cross_entropy ile Dogrulama

In [7]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()
loss_pytorch = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(f"PyTorch Cross-Entropy Loss: {loss_pytorch:.4f}")
print(f"Fark: {abs(loss_pytorch - avg_neg_log_probas):.6f}")

PyTorch Cross-Entropy Loss: 10.7940
Fark: 0.000000


## Egzersiz 2: Perplexity Hesaplama

In [8]:
perplexity = torch.exp(loss_pytorch)
print(f"Cross-Entropy Loss: {loss_pytorch:.4f}")
print(f"Perplexity: {perplexity:.2f}")
print(f"\nVocabulary size: {GPT_CONFIG_124M['vocab_size']}")

Cross-Entropy Loss: 10.7940
Perplexity: 48725.82

Vocabulary size: 50257


In [9]:
loss_values = [0.5, 1.0, 2.0,3.28, 5.0, 10.0]
print("Loss -> Perplexity donusumu:")
for loss in loss_values:
    ppl = math.exp(loss)
    print(f"Loss: {loss:5.2f} -> Perplexity: {ppl:12.2f}")

Loss -> Perplexity donusumu:
Loss:  0.50 -> Perplexity:         1.65
Loss:  1.00 -> Perplexity:         2.72
Loss:  2.00 -> Perplexity:         7.39
Loss:  3.28 -> Perplexity:        26.58
Loss:  5.00 -> Perplexity:       148.41
Loss: 10.00 -> Perplexity:     22026.47


## Egzersiz 3: Training ve Validation Loss

In [10]:
file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

print(f"Veri uzunlugu: {len(text_data)} karakter")

Veri uzunlugu: 20479 karakter


In [11]:
tokenizer = tiktoken.get_encoding("gpt2")
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

torch.manual_seed(123)
train_loader = create_dataloader_v1(
    train_data, batch_size=2, max_length=256, stride=256,
    drop_last=True, shuffle=True, num_workers=0
)
val_loader = create_dataloader_v1(
    val_data, batch_size=2, max_length=256, stride=256,
    drop_last=False, shuffle=False, num_workers=0
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 9, Val batches: 1


### calc_loss_batch Fonksiyonu

In [12]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for input_batch, target_batch in train_loader:
    break
loss = calc_loss_batch(input_batch, target_batch, model, device)
print(f"Batch loss: {loss.item():.4f}")

Batch loss: 11.0140


### calc_loss_loader Fonksiyonu

In [13]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Training Perplexity: {math.exp(train_loss):.2f}")
print(f"Validation Perplexity: {math.exp(val_loss):.2f}")

Training Loss: 10.9876
Validation Loss: 10.9811
Training Perplexity: 59135.32
Validation Perplexity: 58753.49


## Egzersiz 4: Pratik - Loss Monitoring

In [14]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004)

print("Egitim Dongusu - Loss ve Perplexity Takibi")
print("=" * 50)

for step, (input_batch, target_batch) in enumerate(train_loader):
    if step >= 20:
        break
    optimizer.zero_grad()
    loss = calc_loss_batch(input_batch, target_batch, model, device)
    loss.backward()
    optimizer.step()
    if step % 5 == 0:
        print(f"Step {step:3d}: Loss = {loss.item():.4f}, Perplexity = {math.exp(loss.item()):.2f}")

Egitim Dongusu - Loss ve Perplexity Takibi
Step   0: Loss = 11.0262, Perplexity = 61461.25
Step   5: Loss = 8.5537, Perplexity = 5185.86


## Ozet

- **Cross-Entropy Loss**: Modelin tahminlerinin gercek hedeflerden farkini olcer
- **Perplexity**: exp(cross_entropy) - modelin her adimda kac kelime arasinda kararsiz oldugunu gosterir
- Dusuk perplexity = Daha iyi model